In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import json
from datetime import datetime

import hyperparameters
from utils import RandomTrial, read_data

In [2]:
from modelsnnpc_withproba import *

In [3]:
DATASET_LIST = ["Variant I"]
# DATASET_LIST = ["Base", "Variant I", "Variant II", "Variant III", "Variant IV", "Variant V"]
NUM_TRIALS = 10
BEGIN_TRIAL = 0
BASE_SEED = 42

METRICS_NAME_GLOBAL = ["accuracy", "precision", "recall", "fpr", "f1_score","auc"]
METRICS_NAME_5FPR = ["accuracy@5FPR","precision@5FPR", "recall@5FPR", "fpr@5FPR", "f1_score@5FPR"]
METRICS_FAIRNESS = ["fpr_ratio_age", "fpr_ratio_income", "fpr_ratio_employment",
                   "eod_age", "eod_income", "eod_employment",
                   "aod_age", "aod_income", "aod_employment"]

with open("/kaggle/input/get-best-hyperparams/model_2_hyperparameters_selected.json", "r") as file:
    HYPERPARAMETERS = json.load(file)[0]

EXPERIMENT_NAME = f"P{HYPERPARAMETERS['population']}-S{HYPERPARAMETERS['step']}-{NUM_TRIALS}trials-begin{BEGIN_TRIAL}"
FIXED_DATE = None

In [4]:
HYPERPARAMETERS

{'number': 2,
 'values_0': 0.0212644234,
 'values_1': 0.2710215427,
 'datetime_start': '2025-04-05 19:29:48.941809',
 'datetime_complete': '2025-04-05 19:55:16.812815',
 'duration': '0 days 00:25:27.871006',
 'adam_beta1': 0.983,
 'adam_beta2': 0.97,
 'beta1': 0.2116391007,
 'beta2': 0.1585882963,
 'beta3': 0.4339479965,
 'beta4': 0.9383133996,
 'learning_rate': 5.0609e-05,
 'slope': 33,
 'threshold1': 0.2639648969,
 'threshold2': 0.2881911739,
 'threshold3': 0.8150710641,
 'threshold4': 0.3488068823,
 'weight': 0.9887330368,
 'user_attrs_@5FPR accuracy': 0.9582459392,
 'user_attrs_@5FPR aod_age': 0.0212644234,
 'user_attrs_@5FPR aod_employment': 0.0718924364,
 'user_attrs_@5FPR aod_income': 0.0656880803,
 'user_attrs_@5FPR eod_age': 0.0255406848,
 'user_attrs_@5FPR eod_employment': 0.1223113065,
 'user_attrs_@5FPR eod_income': 0.1116794705,
 'user_attrs_@5FPR fnr_ratio_age': 0.9651128051,
 'user_attrs_@5FPR fnr_ratio_employment': 0.8558473887,
 'user_attrs_@5FPR fnr_ratio_income': 0.8

In [5]:
def dataset_loop(train_dfs, test_dfs, dataset_name, trial_number, seed, path, runs):
    x_train = train_dfs[dataset_name].drop(columns=["fraud_bool"])
    y_train = train_dfs[dataset_name]["fraud_bool"]
    x_test = test_dfs[dataset_name].drop(columns=["fraud_bool"])
    y_test = test_dfs[dataset_name]["fraud_bool"]
    num_classes = len(np.unique(y_train))
    num_features = len(x_train.columns)
    class_weights = (1-HYPERPARAMETERS['weight'], HYPERPARAMETERS['weight'])
    model = ModelSNNPC(
        num_features=num_features,
        num_classes=num_classes,
        class_weights=class_weights,
        betas=HYPERPARAMETERS['beta'],
        slope=HYPERPARAMETERS['slope'],
        thresholds=HYPERPARAMETERS['threshold'],
        population=HYPERPARAMETERS['population'],
        batch_size=HYPERPARAMETERS['batch'],
        num_epochs=HYPERPARAMETERS['epoch'],
        num_steps=HYPERPARAMETERS['step'],
        adam_betas=HYPERPARAMETERS['adam_beta'],
        learning_rate=HYPERPARAMETERS['learning_rate'],
        verbose=0
    )
    model.fit(x_train, y_train)
    predictions, targets, proba = model.predict(x_test, y_test)
    np.savetxt(f"predictions_run{trial_number}.csv", predictions, delimiter=",", fmt="%d") 
    np.savetxt(f"targets_run{trial_number}.csv", targets, delimiter=",", fmt="%d")
    np.savetxt(f"proba_run{trial_number}.csv", proba, delimiter="\t")  
    metrics = model.evaluate(targets, predictions)
    metrics_aequitas = model.evaluate_business_constraint(targets, predictions)
    metrics.update(metrics_aequitas)
    fairness_age = model.evaluate_fairness(x_test, targets, predictions, "customer_age", 50)
    metrics.update({k+"_age": v for k, v in fairness_age.items()})
    fairness_income = model.evaluate_fairness(x_test, targets, predictions, "income", 0.5)
    metrics.update({k+"_income": v for k, v in fairness_income.items()})
    fairness_employement = model.evaluate_fairness(x_test, targets, predictions, "employment_status", 3)
    metrics.update({k+"_employment": v for k, v in fairness_employement.items()})
    results = {}
    results["dataset"] = dataset_name
    results["trial"] = trial_number
    results["seed"] = seed
    for metric in METRICS_NAME_GLOBAL:
        results[metric] = metrics[metric]
    for metric in METRICS_NAME_5FPR:
        results[metric] = metrics_aequitas[metric]
    for metric in METRICS_FAIRNESS:
        results[metric] = metrics[metric]
    csv_row = ','.join([str(x) for x in results.values()])
    with open(path, "a") as f:
        f.write(f"{csv_row}\n")
    prev_runs = runs.get(dataset_name, [])
    prev_runs.append(results)
    print(results)
    runs[dataset_name] = prev_runs
    return runs


In [6]:
def simulation(datasets, train_dfs, test_dfs, path="./results.csv"):
    np.random.seed(BASE_SEED)
    seeds = np.random.choice(list(range(1_000_000)), size=NUM_TRIALS, replace=False)
    runs = {}
    for trial in range(NUM_TRIALS):
        seed = seeds[trial]
        trial_number = trial
        trial = RandomTrial(seed=seed)
        if trial_number < BEGIN_TRIAL:
            print(f"Skipping trial {trial_number} – seed {seed}")
            continue
        for dataset_name in datasets.keys():
            print(f"Running trial {trial_number} with seed {seed} on dataset {dataset_name}")
            runs = dataset_loop(train_dfs, test_dfs, dataset_name, trial_number, seed, path, runs)
    return runs

In [7]:
base_path = "/kaggle/input/bank-account-fraud-dataset/"
_, datasets, train_dfs, test_dfs = read_data(base_path, DATASET_LIST)
if not FIXED_DATE:
    date = datetime.now().strftime("%Y%m%d_%H%M%S")
else:
    date = FIXED_DATE
experiment_dir = f"/kaggle/working/results/{date}-{EXPERIMENT_NAME}"
results_path = f"{experiment_dir}/results.csv"
os.makedirs(experiment_dir, exist_ok=True)
if not os.path.exists(results_path):
    with open(results_path, "w") as f:
        f.write("dataset,trial,seed,accuracy,precision,recall,fpr,f1_score,auc,accuracy@5FPR,precision@5FPR,recall@5FPR,fpr@5FPR,f1_score@5FPR,fpr_ratio_age,fpr_ratio_income,fpr_ratio_employment,eod_age,eod_income,eod_employment,aod_age,aod_income,aod_employment\n")

In [8]:
simulation(datasets, train_dfs, test_dfs, path=results_path)

Running trial 0 with seed 987231 on dataset Variant I
{'dataset': 'Variant I', 'trial': 0, 'seed': 987231, 'accuracy': 0.9261401882834983, 'precision': 0.08589951377633712, 'recall': 0.4419735927727589, 'fpr': 0.06696614093760513, 'f1_score': 0.1438425873572317, 'auc': 0.6875037259175768, 'accuracy@5FPR': 0.9859616604068094, 'precision@5FPR': 0, 'recall@5FPR': 0.0, 'fpr@5FPR': 0.0, 'f1_score@5FPR': 0, 'fpr_ratio_age': 0, 'fpr_ratio_income': 0, 'fpr_ratio_employment': 0, 'eod_age': 0.0, 'eod_income': 0.0, 'eod_employment': 0.0, 'aod_age': 0.0, 'aod_income': 0.0, 'aod_employment': 0.0}
Running trial 1 with seed 79954 on dataset Variant I
{'dataset': 'Variant I', 'trial': 1, 'seed': 79954, 'accuracy': 0.8450075606067997, 'precision': 0.05837943576733808, 'recall': 0.6636553161917998, 'fpr': 0.15241030613658402, 'f1_score': 0.10731844360162944, 'auc': 0.7556225050276079, 'accuracy@5FPR': 0.9859616604068094, 'precision@5FPR': 0, 'recall@5FPR': 0.0, 'fpr@5FPR': 0.0, 'f1_score@5FPR': 0, 'fpr_

{'Variant I': [{'dataset': 'Variant I',
   'trial': 0,
   'seed': 987231,
   'accuracy': 0.9261401882834983,
   'precision': 0.08589951377633712,
   'recall': 0.4419735927727589,
   'fpr': 0.06696614093760513,
   'f1_score': 0.1438425873572317,
   'auc': 0.6875037259175768,
   'accuracy@5FPR': 0.9859616604068094,
   'precision@5FPR': 0,
   'recall@5FPR': 0.0,
   'fpr@5FPR': 0.0,
   'f1_score@5FPR': 0,
   'fpr_ratio_age': 0,
   'fpr_ratio_income': 0,
   'fpr_ratio_employment': 0,
   'eod_age': 0.0,
   'eod_income': 0.0,
   'eod_employment': 0.0,
   'aod_age': 0.0,
   'aod_income': 0.0,
   'aod_employment': 0.0},
  {'dataset': 'Variant I',
   'trial': 1,
   'seed': 79954,
   'accuracy': 0.8450075606067997,
   'precision': 0.05837943576733808,
   'recall': 0.6636553161917998,
   'fpr': 0.15241030613658402,
   'f1_score': 0.10731844360162944,
   'auc': 0.7556225050276079,
   'accuracy@5FPR': 0.9859616604068094,
   'precision@5FPR': 0,
   'recall@5FPR': 0.0,
   'fpr@5FPR': 0.0,
   'f1_score